# 🌿 Google Colab — AI Plant Disease Detector Model Training

### 🧩 Step 1: Open Colab

Go to 👉 [https://colab.research.google.com](https://colab.research.google.com)

Create a **new notebook** and paste the entire code below.

---

### 🚀 **Full Colab Code**


In [ ]:
# ============================================
# 🌿 AI Plant Disease Detection Model Training
# By Rushi + GPT-5
# ============================================

# ✅ STEP 1: Install dependencies
!pip install tensorflow kaggle matplotlib --quiet


In [ ]:
# ✅ STEP 2: Mount Google Drive (to save model easily)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ✅ STEP 3: Download the PlantVillage Dataset from Kaggle
# -----------------------------------------------
# 👉 Get your Kaggle API key:
# Go to https://www.kaggle.com > Account > Create API Token
# It will download a file: kaggle.json
# Upload it in the next cell
# -----------------------------------------------

from google.colab import files
files.upload() # Upload your kaggle.json file here


In [ ]:
# ✅ STEP 4: Configure Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
# ✅ STEP 5: Download PlantVillage dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/


In [ ]:
# ✅ STEP 6: Extract dataset
import zipfile
with zipfile.ZipFile("/content/new-plant-diseases-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")


In [ ]:
# ✅ STEP 7: Import libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import os


In [ ]:
# ✅ STEP 8: Prepare training and validation sets
train_dir = "/content/dataset/New Plant Diseases Dataset(Augmented)/train"
valid_dir = "/content/dataset/New Plant Diseases Dataset(Augmented)/valid"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_data = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

num_classes = len(train_data.class_indices)
print(f"✅ Found {num_classes} classes.")


In [ ]:
# ✅ STEP 9: Build Model (Transfer Learning)
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # freeze base layers

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# ✅ STEP 10: Train the Model
EPOCHS = 5 # you can increase to 10-15 for higher accuracy

history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=EPOCHS
)


In [ ]:
# ✅ STEP 11: Save the model
output_path = "/content/drive/MyDrive/plant_disease_model.h5"
model.save(output_path)
print(f"✅ Model saved at: {output_path}")


In [ ]:
# ✅ STEP 12: Save class labels (to use in Flask)
import json

class_labels = list(train_data.class_indices.keys())
with open("/content/drive/MyDrive/class_labels.json", "w") as f:
    json.dump(class_labels, f)

print("✅ Class labels saved successfully!")
